# local-ai-server v0.1.0 — End-to-End Demo

This notebook demonstrates each user-facing endpoint of **local-ai-server v0.1.0** using the OpenAI Python SDK.

**Prerequisites before running:**
- `ollama serve` is running on the host (`http://localhost:11434`).
- Both models are pulled: `ollama pull llama3.1:8b` and `ollama pull nomic-embed-text`.
- The gateway is started in a separate shell: `uv run uvicorn app.main:app --port 8000`
- `api_key` is a placeholder string — auth lands in v0.2.0.

In [ ]:
%pip install openai

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:8000/v1",
    api_key="not-used-yet",
)

## List models

Returns all four registry entries shaped as OpenAI `model` objects.

In [ ]:
models = client.models.list()
for m in models.data:
    print(m.id)

## Non-streaming chat

A single round-trip to Ollama's `/v1/chat/completions`; returns the full completion object.

In [ ]:
r = client.chat.completions.create(
    model="ollama-llama3",
    messages=[{"role": "user", "content": "Say hi in three words."}],
)
print(r.choices[0].message.content)

## Streaming chat (SSE)

The gateway passes Ollama's SSE byte stream through verbatim (including the
terminal `data: [DONE]` frame). The OpenAI SDK consumes the SSE framing and
surfaces each token as a `ChatCompletionChunk`; iteration end signals
the `[DONE]` frame has been received.

In [ ]:
stream = client.chat.completions.create(
    model="ollama-llama3",
    messages=[
        {"role": "user", "content": "Stream a haiku about Mac Studio."}
    ],
    stream=True,
)
parts = []
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        parts.append(chunk.choices[0].delta.content)
print("".join(parts))

## Embeddings

Uses `nomic-embed-text` via Ollama's `/v1/embeddings` endpoint.

In [ ]:
e = client.embeddings.create(
    model="ollama-nomic-embed",
    input="hello world",
)
print(len(e.data[0].embedding))

## 501 from a stub backend (MLX)

In v0.1.0 the MLX adapter is not yet implemented — every method raises
`NotSupportedError` which the gateway translates to HTTP 501. This cell
demonstrates the error surfaces through the OpenAI SDK; the exact exception
class depends on the SDK version (`InternalServerError`, `APIStatusError`, etc.).

In [ ]:
try:
    client.chat.completions.create(
        model="mlx-mistral",
        messages=[{"role": "user", "content": "hi"}],
    )
except Exception as exc:
    print(type(exc).__name__, str(exc)[:120])